In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from libpysal.weights import DistanceBand, lag_spatial
from mgwr.sel_bw import Sel_BW
from mgwr.gwr import GWR
import geopandas as gpd


def get_aicc_safe(fit):
    for name in ("aicc", "AICc", "aic_c"):
        if hasattr(fit, name):
            try:
                return float(getattr(fit, name))
            except Exception:
                pass
    if hasattr(fit, "aic"):
        try:
            return float(getattr(fit, "aic"))
        except Exception:
            pass
    return np.nan

# --- inputs you already have ---
# fin_m : GeoDataFrame with columns design_cols + "incidence"
# coords_all : (n,2) centroids in meters (EPSG:32633)
# design_cols : your predictors list
# distances : e.g. list(range(100_000, 200_001, 10_000))
# kernels : ["gaussian","bisquare","exponential"]

In [4]:
g  = gpd.read_file("../Data/Shapefiles/APES_metric.gpkg", layer="nuts3")
# Replico i filtri: geometrie non vuote e RecordType != "ZIKV"
g = g.loc[(~g.geometry.is_empty) & (g["RecordType"] != "ZIKV")].copy()

# ------------------------------------------------------------------
# 3) PREPARA DATASET come in R (selezione variabili e aggregazione)
# ------------------------------------------------------------------
avg_var = ["Forest", "Shrub", "Urban", "Crop", "Water", "Other",
           "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
           "Mean_T_Spring", "Max_DD_Spring",
           "Mean_T_Summer",
           "Max_WD_Autumn", "Max_DD_Autumn",
           "gdp", "pop_dens"]

cols_keep = (["incidence"] + avg_var +
             ["geometry", "NUTS_NAME", "NUTS_CODE"])

df = g[cols_keep].copy()

# In R raggruppi per geometry, NUTS_NAME, NUTS_CODE; in GeoPandas
# conviene usare l'identificativo NUTS_CODE (che è unico) e conservare la prima geometria.
agg_dict = {"incidence": "sum"}
agg_dict.update({v: "mean" for v in avg_var})
agg_dict.update({"geometry": "first", "NUTS_NAME": "first"})

fin = (
    df.groupby("NUTS_CODE", as_index=False)
      .agg(agg_dict)
      .set_geometry("geometry")
)
fin = gpd.GeoDataFrame(fin, geometry="geometry", crs=g.crs)

# ------------------------------------------------------------------
# 4) STANDARDIZZA (z-score) avg_var
# ------------------------------------------------------------------
num_cols = fin.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
fin_scaled = fin.copy()
fin_scaled[avg_var] = scaler.fit_transform(fin_scaled[avg_var])

In [5]:
kernels = ["gaussian", "bisquare", "exponential"]   # add/remove kernels as you like
thresholds = list(range(100_000, 200_001, 10_000))   # 100–200 km by 10 km
eps = 1e-9

In [6]:
fin_m = fin_scaled.to_crs(32633)
coords_all = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y])
def get_aicc_safe(fit):
    # Try common names across mgwr versions
    for name in ("aicc", "AICc", "aic_c"):
        if hasattr(fit, name):
            try:
                return float(getattr(fit, name))
            except Exception:
                pass
    # Fallback to AIC if nothing else exists
    if hasattr(fit, "aic"):
        try:
            return float(getattr(fit, "aic"))
        except Exception:
            pass
    return np.nan

rows = []
best = None

In [8]:
fin_m.head()

,NUTS_CODE,incidence,Forest,Shrub,Urban,Crop,Water,Other,Mean_P_Winter,Max_WD_Winter,Max_DD_Winter,Mean_T_Spring,Max_DD_Spring,Mean_T_Summer,Max_WD_Autumn,Max_DD_Autumn,gdp,pop_dens,geometry,NUTS_NAME
0,AL011,0.000000,0.756189,0.509149,-0.026845,0.043578,0.086687,0.775878,3.112144,1.945627,-0.714194,-0.550112,-1.092503,-0.157436,0.758760,0.285802,-1.400183,-0.341147,"MULTIPOLYGON (((959466.125 4647785.256, 957657...",Dibër
1,AL012,0.000000,-0.619313,-0.208217,-0.565779,-0.567606,-0.268374,-0.608400,3.359616,1.647878,-0.206647,1.174900,0.627053,1.258868,0.349179,0.488145,-1.341374,-0.025330,"MULTIPOLYGON (((901786.325 4612638.053, 900823...",Durrës
2,AL014,0.000000,-0.093136,-0.029470,-0.644679,-0.524828,-0.114005,-0.561701,3.259787,1.399755,-0.511175,0.473871,-0.101889,0.709290,0.553970,0.488145,-1.401131,-0.314489,"MULTIPOLYGON (((931103.131 4665312.244, 930901...",Lezhë
3,AL015,0.000073,0.511414,0.342044,-0.021393,-0.275947,1.519490,-0.158081,8.237140,2.636556,-0.735017,0.912847,-0.712936,0.838336,0.262537,0.425886,-1.396199,-0.333573,"MULTIPOLYGON (((895277.491 4713536.256, 893963...",Shkodër
4,AL021,0.000000,1.286983,0.588713,-0.098131,0.813643,-0.189353,0.247437,1.844104,0.804258,-0.443502,-0.077223,-0.457015,0.341495,-0.060402,0.892831,-1.407771,-0.306361,"MULTIPOLYGON (((959579.623 4588412.508, 959183...",Elbasan


In [10]:
fin_m_ok = fin_m[avg_var]

In [11]:
design_cols = avg_var

In [12]:
for d in thresholds:
    # 1) Build Y_weighted EXACTLY like your OLS at this distance
    Wd = DistanceBand(coords_all, threshold=d, binary=True, silence_warnings=True)
    y_raw = fin_m["incidence"].to_numpy().reshape(-1, 1)
    y_lag = Wd.sparse.dot(y_raw).ravel()

    y_mean = y_lag.mean()
    y_std  = y_lag.std(ddof=0)
    if y_std == 0:
        print(f"[WARN] {thr/1000:.0f} km → zero variance in Wy; skipping.")
        continue
    y_lag_z = (y_lag - y_mean) / y_std
    
    Xy_thr = fin_m_ok.copy()
    Xy_thr["Y_weighted"] = y_lag_z
    #Xy_thr = pd.DataFrame(tmp.drop(columns=[c for c in ["geometry", "cent"] if c in tmp.columns]))

    # 2) Complete-case mask and align arrays
    need_cols = ["Y_weighted"] + design_cols
    mask = Xy_thr[need_cols].notna().all(axis=1).to_numpy()
    if mask.sum() == 0:
        print(f"[WARN] {d/1000:.0f} km: no complete cases. Skipping.")
        continue

    coords = coords_all[mask]
    y = Xy_thr.loc[mask, "Y_weighted"].to_numpy().reshape(-1, 1)
    X = Xy_thr.loc[mask, design_cols].to_numpy()
    n = X.shape[0]

    row_sums = np.asarray(Wd.sparse.sum(axis=1)).ravel()
    isolated = int((row_sums[mask] == 0).sum())

    for ker in kernels:
        try:
            # Bandwidth search (adaptive neighbor count)
            bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search()

            fit = GWR(coords, y, X, bw,
                      kernel=ker, fixed=False, constant=True, spherical=False).fit()

            aicc_val = get_aicc_safe(fit)
            mean_r2 = float(np.mean(fit.localR2)) if hasattr(fit, "localR2") else np.nan
            min_r2  = float(np.min(fit.localR2))  if hasattr(fit, "localR2") else np.nan
            max_r2  = float(np.max(fit.localR2))  if hasattr(fit, "localR2") else np.nan

            row = {
                "kernel": ker,
                "fixed": False,
                "distance_m_for_Y": d,
                "bandwidth_neighbors": int(bw) if bw is not None else np.nan,
                "AICc": aicc_val,
                "mean_localR2": mean_r2,
                "min_localR2":  min_r2,
                "max_localR2":  max_r2,
                "n": n,
                "isolated_for_Y": isolated
            }
            rows.append(row)

            # Track best by AICc if finite
            if np.isfinite(row["AICc"]) and (best is None or row["AICc"] < best["meta"]["AICc"]):
                best = {"meta": row, "fit": fit, "mask": mask}

            print(f"d={d/1000:.0f} km | {ker:10s} | bw={bw} | "
                  f"AICc={aicc_val if np.isfinite(aicc_val) else 'NA'} | "
                  f"R2 μ/lo/hi={mean_r2:.3f}/{min_r2:.3f}/{max_r2:.3f} | "
                  f"n={n} isoY={isolated}")

        except Exception as e:
            rows.append({
                "kernel": ker, "fixed": False, "distance_m_for_Y": d,
                "bandwidth_neighbors": np.nan, "AICc": np.nan,
                "mean_localR2": np.nan, "min_localR2": np.nan, "max_localR2": np.nan,
                "n": n, "isolated_for_Y": isolated, "error": str(e)
            })
            print(f"d={d/1000:.0f} km | {ker:10s} | ERROR: {e}")

# Save summary
res_df = pd.DataFrame(rows)

d=100 km | gaussian   | bw=75.0 | AICc=2442.936871775326 | R2 μ/lo/hi=-0.685/-41.082/0.790 | n=1134 isoY=27
d=100 km | bisquare   | bw=595.0 | AICc=2634.083297305807 | R2 μ/lo/hi=-1.922/-28.109/0.739 | n=1134 isoY=27
d=100 km | exponential | bw=75.0 | AICc=2361.029791251695 | R2 μ/lo/hi=0.637/0.461/0.771 | n=1134 isoY=27
d=110 km | gaussian   | bw=75.0 | AICc=2345.6470953901216 | R2 μ/lo/hi=-0.788/-45.062/0.814 | n=1134 isoY=11


C:\Users\massaro\AppData\Roaming\Python\Python311\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS


d=110 km | bisquare   | bw=110.0 | AICc=2375.70961886886 | R2 μ/lo/hi=-inf/-inf/0.941 | n=1134 isoY=11
d=110 km | exponential | bw=75.0 | AICc=2283.8962734487254 | R2 μ/lo/hi=0.658/0.468/0.795 | n=1134 isoY=11
d=120 km | gaussian   | bw=75.0 | AICc=2296.173908724132 | R2 μ/lo/hi=-0.738/-45.825/0.821 | n=1134 isoY=7


C:\Users\massaro\AppData\Roaming\Python\Python311\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS


d=120 km | bisquare   | bw=107.0 | AICc=2332.2805246580915 | R2 μ/lo/hi=-inf/-inf/0.943 | n=1134 isoY=7
d=120 km | exponential | bw=75.0 | AICc=2223.424388589609 | R2 μ/lo/hi=0.670/0.476/0.802 | n=1134 isoY=7
d=130 km | gaussian   | bw=75.0 | AICc=2244.9004205409656 | R2 μ/lo/hi=-0.626/-42.309/0.834 | n=1134 isoY=6


...

d=180 km | bisquare   | bw=86.0 | AICc=2064.4427444180665 | R2 μ/lo/hi=-inf/-inf/0.986 | n=1134 isoY=2
d=180 km | exponential | bw=75.0 | AICc=1961.2013595308777 | R2 μ/lo/hi=0.723/0.523/0.850 | n=1134 isoY=2
d=190 km | gaussian   | bw=75.0 | AICc=2029.9386893816154 | R2 μ/lo/hi=-0.061/-20.459/0.878 | n=1134 isoY=2


C:\Users\massaro\AppData\Roaming\Python\Python311\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS


d=190 km | bisquare   | bw=97.0 | AICc=2023.4494331655428 | R2 μ/lo/hi=-inf/-inf/0.985 | n=1134 isoY=2
d=190 km | exponential | bw=75.0 | AICc=1926.0426074540096 | R2 μ/lo/hi=0.729/0.530/0.854 | n=1134 isoY=2
d=200 km | gaussian   | bw=75.0 | AICc=1996.2863123324069 | R2 μ/lo/hi=0.014/-18.318/0.880 | n=1134 isoY=2


C:\Users\massaro\AppData\Roaming\Python\Python311\site-packages\mgwr\gwr.py:775: RuntimeWarning: divide by zero encountered in divide
  return (self.TSS - self.RSS) / self.TSS


d=200 km | bisquare   | bw=108.0 | AICc=1983.4820265358194 | R2 μ/lo/hi=-inf/-inf/0.979 | n=1134 isoY=2
d=200 km | exponential | bw=75.0 | AICc=1894.2902567243368 | R2 μ/lo/hi=0.734/0.539/0.857 | n=1134 isoY=2


In [ ]:
# ======================
# Inputs prep (from file)
# ======================
import geopandas as gpd
import numpy as np
import pandas as pd

# --- file & layer ---
src_path  = "../Data/Shapefiles/APES_metric.gpkg"
src_layer = "nuts3"

# --- identifiers & variable names ---
id_cols       = ["NUTS_CODE", "NUTS_NAME"]          # group-by keys
response_col  = "incidence"                          # dependent variable before lag
predictor_cols = [
    "Forest","Shrub","Urban","Crop","Water","Other",
    "Mean_P_Winter","Max_WD_Winter","Max_DD_Winter",
    "Mean_T_Spring","Max_DD_Spring",
    # "Mean_T_Summer",  # <- include if you want it in the model
    "Max_WD_Autumn","Max_DD_Autumn",
    "gdp","pop_dens"
]

# --- read & basic filter (as you did) ---
gdf = gpd.read_file(src_path, layer=src_layer)
gdf = gdf.loc[(~gdf.geometry.is_empty) & (gdf["RecordType"] != "ZIKV")].copy()

# --- keep only needed columns ---
keep = [c for c in (id_cols + predictor_cols + [response_col, "geometry"]) if c in gdf.columns]
gdf  = gdf[keep].copy()

# --- aggregate to NUTS level ---
# response = sum; predictors = mean
agg_dict = {response_col: "sum"}
agg_dict.update({c: "mean" for c in predictor_cols if c in gdf.columns})
fin = gdf.groupby(id_cols, as_index=False).agg(agg_dict)

# restore a representative geometry per NUTS (first)
geom_first = gdf[id_cols + ["geometry"]].drop_duplicates(id_cols)
fin = fin.merge(geom_first, on=id_cols, how="left")
fin = gpd.GeoDataFrame(fin, geometry="geometry", crs=gdf.crs)

# --- drop rows with NA in variables we need ---
needed = [response_col] + [c for c in predictor_cols if c in fin.columns]
fin = fin.dropna(subset=needed).copy()

# --- project to metric CRS (UTM 33N) and compute centroids ---
fin_m = fin.to_crs(32633)
centroids = fin_m.geometry.centroid
coords_all = np.c_[centroids.x, centroids.y].astype(float)

# (optional) tiny jitter if you have exact duplicate centroids
if np.unique(coords_all, axis=0).shape[0] < coords_all.shape[0]:
    rng = np.random.default_rng(42)
    coords_all = coords_all + rng.normal(0.0, 1e-6, size=coords_all.shape)

# --- final “inputs you already have” ---
design_cols = predictor_cols                              # predictors list
distances   = list(range(100_000, 200_001, 10_000))       # 100–200 km (meters)
kernels     = ["gaussian", "bisquare", "exponential"]     # kernels to test

# fin_m : GeoDataFrame with design_cols + 'incidence'
# coords_all : (n,2) centroid array in meters (EPSG:32633)


In [ ]:
rows = []
best = None

for d in distances:  # d must be in meters (100000..200000)
    # --- 1) Build Wy EXACTLY like before: BINARY SUM (no row-standardization) ---
    Wd = DistanceBand(coords_all, threshold=d, binary=True, silence_warnings=True)

    y_raw = fin_m["incidence"].to_numpy().reshape(-1, 1)
    y_lag = Wd.sparse.dot(y_raw).ravel()         # SUM of neighbors (same as your old code)

    # z-score Wy across all units (same as your previous code)
    y_mean = y_lag.mean()
    y_std  = y_lag.std(ddof=0)
    if y_std == 0:
        print(f"[WARN] {d/1000:.0f} km → zero variance in Wy; skipping.")
        continue
    y_lag_z = (y_lag - y_mean) / y_std

    # --- 2) Assemble frame and complete-case mask (align arrays) ---
    tmp = fin_m.copy()
    tmp["Y_weighted"] = y_lag_z
    Xy_thr = pd.DataFrame(tmp.drop(columns=[c for c in ["geometry", "cent"] if c in tmp.columns]))

    need_cols = ["Y_weighted"] + design_cols
    mask = Xy_thr[need_cols].notna().all(axis=1).to_numpy()
    if mask.sum() == 0:
        print(f"[WARN] {d/1000:.0f} km: no complete cases. Skipping.")
        continue

    coords = coords_all[mask]
    y = Xy_thr.loc[mask, "Y_weighted"].to_numpy().reshape(-1, 1)
    X = Xy_thr.loc[mask, design_cols].to_numpy()
    n, k = X.shape

    # --- 3) Bandwidth selection with a safe minimum to avoid singular locals ---
    bw_min = max(k + 1 + 5, 30)     # params + cushion; at least ~30 neighbors
    bw_min = min(bw_min, n - 1)

    for ker in kernels:
        try:
            bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search(bw_min=bw_min)
            if isinstance(bw, (int, float)) and bw < bw_min:
                bw = bw_min

            fit = GWR(coords, y, X, bw, kernel=ker, fixed=False, constant=True, spherical=False).fit()

            aicc_val = get_aicc_safe(fit)
            mean_r2 = float(np.mean(fit.localR2)) if hasattr(fit, "localR2") else np.nan
            min_r2  = float(np.min(fit.localR2))  if hasattr(fit, "localR2") else np.nan
            max_r2  = float(np.max(fit.localR2))  if hasattr(fit, "localR2") else np.nan

            row = {
                "kernel": ker,
                "fixed": False,
                "distance_m_for_Y": d,
                "bandwidth_neighbors": int(bw) if bw is not None else np.nan,
                "AICc": aicc_val,
                "mean_localR2": mean_r2,
                "min_localR2":  min_r2,
                "max_localR2":  max_r2,
                "n": int(n),
                # islands not dropped in this reproduction to match your earlier behavior:
                "isolated_for_Y": int((np.asarray(Wd.sparse.sum(axis=1)).ravel()[mask] == 0).sum()),
            }
            rows.append(row)

            if np.isfinite(row["AICc"]) and (best is None or row["AICc"] < best["meta"]["AICc"]):
                best = {"meta": row, "fit": fit, "mask": mask}

            print(f"d={d/1000:.0f} km | {ker:10s} | bw={row['bandwidth_neighbors']} | "
                  f"AICc={aicc_val if np.isfinite(aicc_val) else 'NA'} | "
                  f"R2 μ/lo/hi={mean_r2:.3f}/{min_r2:.3f}/{max_r2:.3f} | n={n}")

        except Exception as e:
            rows.append({
                "kernel": ker, "fixed": False, "distance_m_for_Y": d,
                "bandwidth_neighbors": np.nan, "AICc": np.nan,
                "mean_localR2": np.nan, "min_localR2": np.nan, "max_localR2": np.nan,
                "n": int(n), "isolated_for_Y": None, "error": str(e)
            })
            print(f"d={d/1000:.0f} km | {ker:10s} | ERROR: {e}")


In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from libpysal.weights import DistanceBand, lag_spatial
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW

g  = gpd.read_file("../Data/Shapefiles/APES_metric.gpkg", layer="nuts3")
g = g.loc[(~g.geometry.is_empty) & (g["RecordType"] != "ZIKV")].copy()
def get_aicc_safe(fit):
    # Try common names across mgwr versions
    for name in ("aicc", "AICc", "aic_c"):
        if hasattr(fit, name):
            try:
                return float(getattr(fit, name))
            except Exception:
                pass
    # Fallback to AIC if nothing else exists
    if hasattr(fit, "aic"):
        try:
            return float(getattr(fit, "aic"))
        except Exception:
            pass
    return np.nan

design_cols = ["Forest", "Shrub", "Urban", "Crop", "Water", "Other",
           "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
           "Mean_T_Spring", "Max_DD_Spring",
           "Mean_T_Summer",
           "Max_WD_Autumn", "Max_DD_Autumn",
           "gdp", "pop_dens"]

cols_keep = (["incidence"] + design_cols + ["geometry", "NUTS_NAME", "NUTS_CODE"])

df = g[cols_keep].copy()
agg_dict = {"incidence": "sum"}
agg_dict.update({v: "mean" for v in design_cols})
agg_dict.update({"geometry": "first", "NUTS_NAME": "first"})

fin = (df.groupby("NUTS_CODE", as_index=False).agg(agg_dict).set_geometry("geometry"))
fin = gpd.GeoDataFrame(fin, geometry="geometry", crs=g.crs)
fin_m = fin.to_crs(32633)
coords_all = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y])

In [ ]:
# --- Minimal, robust GWR @ 150 km (Gaussian) ---
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from libpysal.weights import DistanceBand, lag_spatial
from mgwr.sel_bw import Sel_BW
from mgwr.gwr import GWR

# ---------- 0) READ + PREP GEODATA ----------
g  = gpd.read_file("../Data/Shapefiles/APES_metric.gpkg", layer="nuts3")
g  = g.loc[(~g.geometry.is_empty) & (g["RecordType"] != "ZIKV")].copy()

design_cols = [
    "Forest","Shrub","Urban","Crop","Water","Other",
    "Mean_P_Winter","Max_WD_Winter","Max_DD_Winter",
    "Mean_T_Spring","Max_DD_Spring",
    # "Mean_T_Summer",   # uncomment to include it
    "Max_WD_Autumn","Max_DD_Autumn",
    "gdp","pop_dens"
]
predictor_cols = design_cols
gdf = g.copy()
id_cols = ["NUTS_NAME", "NUTS_CODE"]
response_col = 'incidence'
keep = [c for c in (id_cols + design_cols + [response_col, "geometry"]) if c in gdf.columns]
gdf = gdf[keep].copy()

# 2) Aggregate to NUTS (your pattern): response=sum; predictors=mean
agg_dict = {response_col: "sum"}
agg_dict.update({c: "mean" for c in design_cols if c in gdf.columns})
fin_eu3 = gdf.groupby(id_cols, as_index=False).agg(agg_dict)

# representative geometry per NUTS (your pattern)
geom_first = gdf[id_cols + ["geometry"]].drop_duplicates(id_cols)
fin_eu3 = fin_eu3.merge(geom_first, on=id_cols, how="left")
fin_eu3 = gpd.GeoDataFrame(fin_eu3, geometry="geometry", crs=gdf.crs)

# drop rows with NA in variables we need (your pattern)
needed = [response_col] + [c for c in design_cols if c in fin_eu3.columns]
fin_eu3 = fin_eu3.dropna(subset=needed).copy()

# 3) Project to metric CRS and centroids (your pattern)
fin_eu3 = fin_eu3.to_crs(32633)
centroids = fin_eu3.geometry.centroid
coords_all = np.c_[centroids.x, centroids.y].astype(float)

# tiny jitter if exact duplicate centroids
if np.unique(coords_all, axis=0).shape[0] < coords_all.shape[0]:
    rng = np.random.default_rng(42)
    coords_all = coords_all + rng.normal(0.0, 1e-6, size=coords_all.shape)

In [ ]:
# 4) Build row-std W at 150 km and spatial lag of y, then z-score AFTER lag
d_m = 150_000
W = DistanceBand(coords_all, threshold=d_m, binary=True, silence_warnings=True)
W.transform = "R"

y_raw = fin_eu3[response_col].to_numpy().astype(float)
Wy = lag_spatial(W, y_raw)

deg = np.asarray(W.sparse.sum(axis=1)).ravel()
mask_islands = deg > 0  # drop isolates for stability

Wy_mean = Wy[mask_islands].mean()
Wy_std  = Wy[mask_islands].std(ddof=0)
Wy_z    = (Wy - Wy_mean) / Wy_std

# 5) Assemble modeling frame, clean mask, and standardize X on used rows
Xy = fin_eu3[design_cols].copy()
Xy["Y_weighted"] = Wy_z

mask_cc = Xy[["Y_weighted"] + design_cols].notna().all(axis=1).to_numpy()
mask = mask_cc & mask_islands & np.isfinite(coords_all).all(axis=1)

In [ ]:
coords = coords_all[mask]
y = Xy.loc[mask, "Y_weighted"].to_numpy().reshape(-1, 1)

x_scaler = StandardScaler().fit(Xy.loc[mask, predictor_cols])
X = x_scaler.transform(Xy.loc[mask, predictor_cols])

# 6) Bandwidth selection and GWR fit (Gaussian kernel; adaptive)
n, k = X.shape
bw_min = max(k + 1 + 5, 30)  # params + cushion; ensure ≥ ~30 neighbors
bw_min = min(bw_min, n - 1)

bw = Sel_BW(coords, y, X, kernel="gaussian", fixed=False, spherical=False).search(bw_min=bw_min)
if isinstance(bw, (int, float)) and bw < bw_min:
    bw = bw_min

fit = GWR(coords, y, X, bw, kernel="gaussian", fixed=False, constant=True, spherical=False).fit()

# 7) Quick summary
def get_aicc_safe(f):
    for nm in ("aicc","AICc","aic_c"):
        if hasattr(f, nm):
            try: return float(getattr(f, nm))
            except Exception: pass
    return float(getattr(f, "aic", np.nan)) if hasattr(f, "aic") else np.nan

print(f"kernel=gaussian, distance=150 km, bw={int(bw)}, n={n}")
print(f"AICc={get_aicc_safe(fit):.2f} | localR2 mean={fit.localR2.mean():.3f} "
      f"min={fit.localR2.min():.3f} max={fit.localR2.max():.3f}")

# 8) (Optional) export local params for mapping
coef_names = ["Intercept"] + predictor_cols
betas = pd.DataFrame(fit.params,  columns=coef_names)
tvals = pd.DataFrame(fit.tvalues, columns=[f"t_{c}" for c in coef_names])
locR2 = pd.Series(fit.localR2, name="localR2")

gout = fin_eu3.loc[mask].copy().reset_index(drop=True)

In [ ]:
# ---------- 1) CHOOSE DISTANCE + KERNEL ----------
d_m   = 150_000                   # 150 km
ker   = "gaussian"

# ---------- 2) BUILD ROW-STANDARDIZED W + Wy (z-score AFTER lag) ----------
W = DistanceBand(coords_all, threshold=d_m, binary=True, silence_warnings=True)
W.transform = "R"

y_raw = fin["incidence"].to_numpy().astype(float)
Wy    = lag_spatial(W, y_raw)                         # neighbor AVERAGE thanks to transform='R'

# drop islands (deg == 0) for stability
deg          = np.asarray(W.sparse.sum(axis=1)).ravel()
mask_island  = deg > 0

# z-score Wy using only non-islands (prevents NaN/inf)
Wy_mean = Wy[mask_island].mean()
Wy_std  = Wy[mask_island].std(ddof=0)
Wy_z    = (Wy - Wy_mean) / Wy_std

# ---------- 3) ASSEMBLE MODEL DATA + CLEAN MASK ----------
X_df = fin[design_cols].copy()
Xy   = X_df.copy()
Xy["Y_weighted"] = Wy_z

mask_cc = Xy[["Y_weighted"] + design_cols].notna().all(axis=1).to_numpy()
mask    = mask_cc & mask_island & np.isfinite(coords_all).all(axis=1)

coords = coords_all[mask]
y      = Xy.loc[mask, "Y_weighted"].to_numpy().reshape(-1, 1)

# standardize X **on the rows used in the fit**
x_scaler = StandardScaler().fit(Xy.loc[mask, design_cols])
X       = x_scaler.transform(Xy.loc[mask, design_cols])

n, k = X.shape

In [ ]:
# ---------- 4) BANDWIDTH SELECTION (adaptive) ----------
bw_min = max(k + 1 + 5, 30)         # params + small cushion; at least ~30 neighbors
bw_min = min(bw_min, n - 1)

bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search(bw_min=bw_min)
if isinstance(bw, (int, float)) and bw < bw_min:
    bw = bw_min

# ---------- 5) FIT GWR ----------
fit = GWR(coords, y, X, bw, kernel=ker, fixed=False, constant=True, spherical=False).fit()

# ---------- 6) QUICK SUMMARY ----------
def get_aicc_safe(f):
    for name in ("aicc","AICc","aic_c"):
        if hasattr(f, name):
            try: return float(getattr(f, name))
            except Exception: pass
    return float(getattr(f, "aic", np.nan)) if hasattr(f, "aic") else np.nan

print(f"kernel={ker}, distance={d_m/1000:.0f} km, bw(adaptive nn)={int(bw)}")
print(f"AICc={get_aicc_safe(fit):.2f} | localR2 mean={fit.localR2.mean():.3f} "
      f"min={fit.localR2.min():.3f} max={fit.localR2.max():.3f} | n={n}")

# ---------- 7) (OPTIONAL) EXPORT LOCAL RESULTS FOR MAPPING ----------
coef_names = ["Intercept"] + design_cols
betas = pd.DataFrame(fit.params,   columns=coef_names)
tvals = pd.DataFrame(fit.tvalues,  columns=[f"t_{c}" for c in coef_names])
locR2 = pd.Series(fit.localR2, name="localR2")

gout = fin.loc[mask].copy().reset_index(drop=True)
gout = gpd.GeoDataFrame(pd.concat([gout, betas, tvals, locR2], axis=1),
                        geometry="geometry", crs=fin.crs)

# gout.to_file("../Results/gwr_150km_gaussian.gpkg", layer="gwr_150km_gaussian", driver="GPKG")
# print("Saved: ../Results/gwr_150km_gaussian.gpkg : layer 'gwr_150km_gaussian'")

In [ ]:
### Example deistance 150km, kernel Gaussian
ker='gaussian'
d = 150000

Wd = DistanceBand(coords_all, threshold=d, binary=True, silence_warnings=True)
y_raw = fin_m["incidence"].to_numpy().reshape(-1, 1)
y_lag = Wd.sparse.dot(y_raw).ravel()
y_mean = y_lag.mean()
y_std  = y_lag.std(ddof=0)
if y_std == 0:
    print(f"[WARN] {d/1000:.0f} km → zero variance in Wy; skipping.")
y_lag_z = (y_lag - y_mean) / y_std

tmp = fin_m.copy()
tmp["Y_weighted"] = y_lag_z
Xy_thr = pd.DataFrame(tmp.drop(columns=[c for c in ["geometry", "cent"] if c in tmp.columns]))
# 2) Complete-case mask and align arrays
need_cols = ["Y_weighted"] + design_cols
mask = Xy_thr[need_cols].notna().all(axis=1).to_numpy()
if mask.sum() == 0:
    print(f"[WARN] {d/1000:.0f} km: no complete cases. Skipping.")

coords = coords_all[mask]
y = Xy_thr.loc[mask, "Y_weighted"].to_numpy().reshape(-1, 1)

scaler = StandardScaler()
Xy_thr[design_cols] = scaler.fit_transform(Xy_thr[design_cols])
X = Xy_thr.loc[mask, design_cols].to_numpy()




In [ ]:
bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search()
fit = GWR(coords, y, X, bw, kernel=ker, fixed=False, constant=True, spherical=False).fit()

In [ ]:
print(f"kernel={ker}, d={d/1000:.0f} km, bw={int(bw)}")
print(f"AICc={getattr(fit, 'aicc', getattr(fit, 'AICc', np.nan)):.2f}, "
      f"localR2 mean={fit.localR2.mean():.3f}, min={fit.localR2.min():.3f}, max={fit.localR2.max():.3f}")

In [ ]:
ker = "gaussian"
d_m = 150_000  # 150 km in meters

# 0) Coords (tiny jitter if duplicates)
coords_all = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y]).astype(float)
if np.unique(coords_all, axis=0).shape[0] < coords_all.shape[0]:
    rng = np.random.default_rng(42)
    coords_all = coords_all + rng.normal(0.0, 1e-6, size=coords_all.shape)

# 1) Build row-std W and Wy (z-score AFTER lag)
W = DistanceBand(coords_all, threshold=d_m, binary=True, silence_warnings=True)
W.transform = "R"
y_raw = fin_m["incidence"].to_numpy().astype(float)      # raw is fine; Wy will be z-scored
Wy = lag_spatial(W, y_raw)

deg = np.asarray(W.sparse.sum(axis=1)).ravel()
mask_islands = deg > 0                                   # drop islands for stability

Wy_mean = Wy[mask_islands].mean()
Wy_std  = Wy[mask_islands].std(ddof=0)
Wy_z = (Wy - Wy_mean) / Wy_std


df = fin_m[design_cols].copy()
df["Y_weighted"] = Wy_z

mask_cc = df[["Y_weighted"] + design_cols].notna().all(axis=1).to_numpy()
mask = mask_cc & mask_islands & np.isfinite(coords_all).all(axis=1)

coords = coords_all[mask]
y = df.loc[mask, "Y_weighted"].to_numpy().reshape(-1, 1)

# 3) Standardize X **on the used rows**
scaler = StandardScaler().fit(df.loc[mask, design_cols])
X = scaler.transform(df.loc[mask, design_cols])

In [ ]:
# 4) Bandwidth selection (set a safe minimum)
n, k = X.shape
bw_min = max(k + 1 + 5, 30)         # params + cushion; at least ~30 neighbors
bw_min = min(bw_min, n - 1)

bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search(bw_min=bw_min)
if isinstance(bw, (int, float)) and bw < bw_min:
    bw = bw_min

fit = GWR(coords, y, X, bw, kernel=ker, fixed=False, constant=True, spherical=False).fit()

print(f"kernel={ker}, d={d_m/1000:.0f} km, bw={int(bw)}")
print(f"AICc={getattr(fit, 'aicc', getattr(fit, 'AICc', np.nan)):.2f}, "
      f"localR2 mean={fit.localR2.mean():.3f}, min={fit.localR2.min():.3f}, max={fit.localR2.max():.3f}")

In [ ]:



rows = []
best = None

for d in distances:
    # 1) Build Y_weighted EXACTLY like your OLS at this distance
    Wd = DistanceBand(coords_all, threshold=d, binary=True, silence_warnings=True)
    y_raw = fin_m["incidence"].to_numpy().reshape(-1, 1)
    y_lag = Wd.sparse.dot(y_raw).ravel()

    y_mean = y_lag.mean()
    y_std  = y_lag.std(ddof=0)
    if y_std == 0:
        print(f"[WARN] {thr/1000:.0f} km → zero variance in Wy; skipping.")
        continue
    y_lag_z = (y_lag - y_mean) / y_std
    
    tmp = fin_m.copy()
    tmp["Y_weighted"] = y_lag_z
    Xy_thr = pd.DataFrame(tmp.drop(columns=[c for c in ["geometry", "cent"] if c in tmp.columns]))

    # 2) Complete-case mask and align arrays
    need_cols = ["Y_weighted"] + design_cols
    mask = Xy_thr[need_cols].notna().all(axis=1).to_numpy()
    if mask.sum() == 0:
        print(f"[WARN] {d/1000:.0f} km: no complete cases. Skipping.")
        continue

    coords = coords_all[mask]
    y = Xy_thr.loc[mask, "Y_weighted"].to_numpy().reshape(-1, 1)
    X = Xy_thr.loc[mask, design_cols].to_numpy()
    n = X.shape[0]

    row_sums = np.asarray(Wd.sparse.sum(axis=1)).ravel()
    isolated = int((row_sums[mask] == 0).sum())

    for ker in kernels:
        try:
            # Bandwidth search (adaptive neighbor count)
            bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search()

            fit = GWR(coords, y, X, bw,
                      kernel=ker, fixed=False, constant=True, spherical=False).fit()

            aicc_val = get_aicc_safe(fit)
            mean_r2 = float(np.mean(fit.localR2)) if hasattr(fit, "localR2") else np.nan
            min_r2  = float(np.min(fit.localR2))  if hasattr(fit, "localR2") else np.nan
            max_r2  = float(np.max(fit.localR2))  if hasattr(fit, "localR2") else np.nan

            row = {
                "kernel": ker,
                "fixed": False,
                "distance_m_for_Y": d,
                "bandwidth_neighbors": int(bw) if bw is not None else np.nan,
                "AICc": aicc_val,
                "mean_localR2": mean_r2,
                "min_localR2":  min_r2,
                "max_localR2":  max_r2,
                "n": n,
                "isolated_for_Y": isolated
            }
            rows.append(row)

            # Track best by AICc if finite
            if np.isfinite(row["AICc"]) and (best is None or row["AICc"] < best["meta"]["AICc"]):
                best = {"meta": row, "fit": fit, "mask": mask}

            print(f"d={d/1000:.0f} km | {ker:10s} | bw={bw} | "
                  f"AICc={aicc_val if np.isfinite(aicc_val) else 'NA'} | "
                  f"R2 μ/lo/hi={mean_r2:.3f}/{min_r2:.3f}/{max_r2:.3f} | "
                  f"n={n} isoY={isolated}")

        except Exception as e:
            rows.append({
                "kernel": ker, "fixed": False, "distance_m_for_Y": d,
                "bandwidth_neighbors": np.nan, "AICc": np.nan,
                "mean_localR2": np.nan, "min_localR2": np.nan, "max_localR2": np.nan,
                "n": n, "isolated_for_Y": isolated, "error": str(e)
            })
            print(f"d={d/1000:.0f} km | {ker:10s} | ERROR: {e}")

# Save summary
res_df = pd.DataFrame(rows)

In [ ]:
design_cols = ["Forest", "Shrub", "Urban", "Crop", "Water", "Other",
           "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
           "Mean_T_Spring", "Max_DD_Spring",
           "Mean_T_Summer",
           "Max_WD_Autumn", "Max_DD_Autumn",
           "gdp", "pop_dens"]

cols_keep = (["incidence"] + design_cols +
             ["geometry", "NUTS_NAME", "NUTS_CODE"])

df = g[cols_keep].copy()
agg_dict = {"incidence": "sum"}
agg_dict.update({v: "mean" for v in avg_var})
agg_dict.update({"geometry": "first", "NUTS_NAME": "first"})

fin = (
    df.groupby("NUTS_CODE", as_index=False)
      .agg(agg_dict)
      .set_geometry("geometry")
)
fin = gpd.GeoDataFrame(fin, geometry="geometry", crs=g.crs)
fin["cent"] = fin.geometry.centroid

In [ ]:
fin_m = fin.to_crs(32633)
coords = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y])

In [ ]:

# 4) Create flat DataFrame for statsmodels (same as your OLS)
def fmt_p(p):
    try:
        return "<.001" if p < 0.001 else f"{p:.3f}"
    except Exception:
        return str(p)
# --- prepare standardized Xs ---
tmp = fin_m.copy()
X_thr = pd.DataFrame(tmp.drop(columns=[c for c in ["geometry", "cent"] if c in tmp.columns]))

design_cols = avg_var + ["gdp", "pop_dens"]

In [ ]:
scaler = StandardScaler()
X_all = fin_m[design_cols]                       # DataFrame
scaler.fit(X_all.dropna())                       # fit on complete rows
X_scaled_all = pd.DataFrame(
    scaler.transform(X_all), 
    columns=design_cols, index=X_all.index
)

coords_all = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y])
y_vec = fin_m["incidence"].to_numpy()            # raw or z, doesn't matter; we z Wy later


In [ ]:
# --- loop thresholds ---
thresholds = np.arange(100_000, 200_000 + 1, 10_000)
kernels = ["gaussian", "bisquare", "exponential"]   # add/remove kernels as you like

y_vec = fin_m["incidence"].to_numpy()            # raw or z, doesn't matter; we z Wy later

rows = []
for thr in thresholds:
    # --- W and Wy ---
    W = DistanceBand(coords_all, threshold=thr, binary=True, silence_warnings=True)
    W.transform = "R"
    Wy = lag_spatial(W, y_vec)

    # Replace possible NaNs in Wy (can happen with islands) with 0 before z-scoring
    Wy = np.asarray(Wy, dtype=float)
    Wy[np.isnan(Wy)] = 0.0

    Wy_mean = Wy.mean()
    Wy_std  = Wy.std(ddof=0)
    if Wy_std == 0:
        print(f"[WARN] {thr//1000} km: Wy has zero variance; skipping.")
        continue
    Wy_z = (Wy - Wy_mean) / Wy_std

    # --- Assemble a modeling frame and make a complete-case mask ---
    df = X_scaled_all.copy()                      # standardized X
    df["Y_weighted"] = Wy_z
    # mask for complete cases in X and y, and finite coords
    mask = df[["Y_weighted"] + design_cols].notna().all(axis=1).to_numpy()
    mask &= np.isfinite(Wy_z)
    mask &= np.isfinite(coords_all).all(axis=1)

    # also remove rows with no neighbors if needed (optional)
    deg = np.asarray(W.sparse.sum(axis=1)).ravel()
    # mask &= deg > 0  # uncomment if you prefer to drop islands instead of setting Wy=0

    if mask.sum() == 0:
        print(f"[WARN] {thr//1000} km: no valid rows after masking; skipping.")
        continue

    coords = coords_all[mask]
    y = Wy_z[mask].reshape(-1, 1)
    X = df.loc[mask, design_cols].to_numpy()

    # --- Quick sanity: print NaN counts if any ---
    n_nan = np.isnan(X).sum() + np.isnan(y).sum() + np.isnan(coords).sum()
    if n_nan > 0:
        print(f"[ERROR] {thr//1000} km: still NaNs after masking (count={n_nan}).")
        continue

    # --- GWR per kernel ---
    for ker in kernels:
        try:
            bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search()
            fit = GWR(coords, y, X, bw, kernel=ker, fixed=False, constant=True, spherical=False).fit()

            rows.append({
                "kernel": ker,
                "fixed": False,
                "max_distance_m": thr,
                "bandwidth_neighbors": int(bw),
                "AICc": float(getattr(fit, "aicc", getattr(fit, "AICc", np.nan))),
                "mean_localR2": float(fit.localR2.mean()),
                "min_localR2":  float(fit.localR2.min()),
                "max_localR2":  float(fit.localR2.max()),
                "n": int(mask.sum()),
            })
            print(f"d={thr//1000} km | {ker:10s} | bw={bw:4} | "
                  f"AICc={rows[-1]['AICc']:.1f} | R2 μ/lo/hi="
                  f"{fit.localR2.mean():.3f}/{fit.localR2.min():.3f}/{fit.localR2.max():.3f} | n={mask.sum()}")

        except Exception as e:
            rows.append({
                "kernel": ker, "fixed": False, "max_distance_m": thr,
                "bandwidth_neighbors": np.nan, "AICc": np.nan,
                "mean_localR2": np.nan, "min_localR2": np.nan, "max_localR2": np.nan,
                "n": int(mask.sum()), "error": str(e)
            })
            print(f"d={thr//1000} km | {ker:10s} | ERROR: {e}")


In [ ]:
design_cols = [
    "Forest","Shrub","Urban","Crop","Water","Other",
    "Mean_P_Winter","Max_WD_Winter","Max_DD_Winter",
    "Mean_T_Spring","Max_DD_Spring",
    "Max_WD_Autumn","Max_DD_Autumn",
    "gdp","pop_dens"
]

kernels    = ["gaussian", "bisquare", "exponential"]
distances  = list(range(100_000, 200_001, 10_000))  # 100–200 km

# Coords (add tiny jitter if there are exact duplicates)
coords_all = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y]).astype(float)
if np.unique(coords_all, axis=0).shape[0] < coords_all.shape[0]:
    rng = np.random.default_rng(42)
    coords_all = coords_all + rng.normal(0.0, 1e-6, size=coords_all.shape)

# Standardize X globally once (clean & reproducible)
X_all = fin_m[design_cols].copy()
x_scaler = StandardScaler().fit(X_all.dropna())
X_scaled_all = pd.DataFrame(x_scaler.transform(X_all), columns=design_cols, index=X_all.index)

# y raw (we'll lag + z-score Wy per distance)
y_raw = fin_m["incidence"].to_numpy().astype(float)

def get_aicc_safe(fit):
    for name in ("aicc", "AICc", "aic_c"):
        if hasattr(fit, name):
            try:
                return float(getattr(fit, name))
            except Exception:
                pass
    if hasattr(fit, "aic"):
        try:
            return float(getattr(fit, "aic"))
        except Exception:
            pass
    return np.nan

rows = []
best = None

# ----------------------------
# Loop over distances
# ----------------------------
for d in distances:
    # 1) Build row-std W and compute Wy
    Wd = DistanceBand(coords_all, threshold=d, binary=True, silence_warnings=True)
    Wd.transform = "R"

    deg = np.asarray(Wd.sparse.sum(axis=1)).ravel()
    # Option A: drop islands (recommended for GWR stability)
    mask_islands = deg > 0

    Wy = lag_spatial(Wd, y_raw)                # neighbor average (thanks to transform='R')
    Wy = np.asarray(Wy, dtype=float)

    # 2) z-score Wy (AFTER lag), but only on non-islands
    Wy_mean = Wy[mask_islands].mean()
    Wy_std  = Wy[mask_islands].std(ddof=0)
    if Wy_std == 0:
        print(f"[WARN] {d//1000} km: Wy has zero variance; skipping.")
        continue
    Wy_z = (Wy - Wy_mean) / Wy_std

    # 3) Build modeling frame and complete-case mask
    df = X_scaled_all.copy()
    df["Y_weighted"] = Wy_z

    # complete cases in X and y AND non-islands AND finite coords
    mask_cc = df[["Y_weighted"] + design_cols].notna().all(axis=1).to_numpy()
    mask_cc &= mask_islands
    mask_cc &= np.isfinite(coords_all).all(axis=1)

    if mask_cc.sum() == 0:
        print(f"[WARN] {d//1000} km: no valid rows after masking; skipping.")
        continue

    coords = coords_all[mask_cc]
    y = df.loc[mask_cc, "Y_weighted"].to_numpy().reshape(-1, 1)
    X = df.loc[mask_cc, design_cols].to_numpy()
    n, k = X.shape

    # 4) Adaptive bandwidth with a safe minimum to avoid singular locals
    bw_min = max(k + 1 + 5, 30)            # params + cushion; ensure ≥ ~30 neighbors
    bw_min = min(bw_min, n - 1)

    # 5) Fit GWR per kernel
    for ker in kernels:
        try:
            bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search(bw_min=bw_min)
            # Enforce bw_min if selector returns smaller
            if isinstance(bw, (int, float)) and bw < bw_min:
                bw = bw_min

            fit = GWR(coords, y, X, bw,
                      kernel=ker, fixed=False, constant=True, spherical=False).fit()

            aicc_val = get_aicc_safe(fit)
            mean_r2  = float(fit.localR2.mean()) if hasattr(fit, "localR2") else np.nan
            min_r2   = float(fit.localR2.min())  if hasattr(fit, "localR2") else np.nan
            max_r2   = float(fit.localR2.max())  if hasattr(fit, "localR2") else np.nan

            row = {
                "kernel": ker,
                "fixed": False,
                "distance_m_for_Y": d,                 # distance used to build Wy
                "bandwidth_neighbors": int(bw),
                "AICc": aicc_val,
                "mean_localR2": mean_r2,
                "min_localR2":  min_r2,
                "max_localR2":  max_r2,
                "n": int(n),
                "dropped_islands": int((~mask_islands).sum())
            }
            rows.append(row)

            # Track best by AICc
            if np.isfinite(aicc_val) and (best is None or aicc_val < best["meta"]["AICc"]):
                best = {"meta": row, "fit": fit, "mask": mask_cc}

            print(f"d={d/1000:.0f} km | {ker:10s} | bw={int(bw):4d} | "
                  f"AICc={aicc_val if np.isfinite(aicc_val) else 'NA'} | "
                  f"R2 μ/lo/hi={mean_r2:.3f}/{min_r2:.3f}/{max_r2:.3f} | n={n}")

        except np.linalg.LinAlgError as e:
            print(f"d={d/1000:.0f} km | {ker:10s} | SINGULAR: {e} (try larger bw or drop more collinear X)")
        except Exception as e:
            rows.append({
                "kernel": ker, "fixed": False, "distance_m_for_Y": d,
                "bandwidth_neighbors": np.nan, "AICc": np.nan,
                "mean_localR2": np.nan, "min_localR2": np.nan, "max_localR2": np.nan,
                "n": int(n), "dropped_islands": int((~mask_islands).sum()),
                "error": str(e)
            })
            print(f"d={d/1000:.0f} km | {ker:10s} | ERROR: {e}")